# Exercise 2: Data, Metadata & Query Engines

In this exercise, you'll compare how different database systems handle data and metadata. You'll observe performance differences and see how traditional databases bundle components versus how modern data catalogs separate them. To get started, imagine you have a dataset of product sales. Note, these could be any item: 

| transaction\_id | product\_id | sale\_date | quantity | price |
| :---: | :---: | :---: | :---: | :---: |
| 1 | 101 | 2025-10-14 | 5 | 19.99 |
| 2 | 102 | 2025-10-14 | 1 | 250.00 |
| 3 | 101 | 2026-01-30 | 3 | 19.99 |
| 4 | 103 | 2026-03-15 | 10 | 5.50 |
| 5 | 102 | 2026-10-16 | 2 | 245.00 |

We'll load this information into DuckDB, as well PostgreSQL to start. Both DuckDB and PostgreSQL are end-to-end full databases. They provide the three main components we need to work with data (this is a simplified worldview, but bear with us):

- Storage
- Metadata 
- Query Engine

It's easy to see why storage matters. That's the core functionality of a database - to store data. Query engines also make intuitive sense at first glance - you have to be able to retreive the data you've stored, otherwise what's the point? If you couldn't get the data back using standard SQL syntax, might as well send the data into a black hole. For that reason, we'll start our journey with these more intuitive components - loading and data storage, and then separate out all the pieces of the database to see what's really going on, and why metadata is also critical.




## Step 1: Data Loading

The following Python code snippets use duckdb for an in-memory database and psycopg2 to connect to a running Postgres database.


### DuckDB:

In [ ]:
%config SqlMagic.autopandas = True;
%config SqlMagic.feedback = False;
%config SqlMagic.displaycon = False;
import duckdb;
import pandas as pd
import zipfile;
import os;
from io import StringIO

%load_ext sql
conn = duckdb.connect();
%sql conn --alias duckdb
%sql duckdb:///:default:

In [72]:
%%sql
CREATE TABLE sales (
    transaction_id INTEGER,
    product_id INTEGER,
    sale_date DATE,
    quantity INTEGER,
    price DOUBLE
);

,Count


In [ ]:
%%timeit
%%sql
INSERT INTO sales (transaction_id, product_id, sale_date, quantity, price) VALUES
(1, 101, '2025-10-14', 5, 19.99),
-- FILL THIS IN,
(3, 101, '2026-01-30', 3, 19.99),
-- FILL THIS IN,
(5, 102, '2026-10-16', 2, 245.00);

6.49 ms ± 170 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [70]:
%%sql
SELECT * FROM sales;

,transaction_id,product_id,sale_date,quantity,price
0,1,101,2025-10-14,5,19.99
1,2,102,2025-10-14,1,250.00
2,3,101,2026-01-30,3,19.99
3,4,103,2026-03-15,10,5.50
4,5,102,2026-10-16,2,245.00
...,...,...,...,...,...
4050,1,101,2025-10-14,5,19.99
4051,2,102,2025-10-14,1,250.00
4052,3,101,2026-01-30,3,19.99
4053,4,103,2026-03-15,10,5.50


## PostgreSQL

In [15]:
import psycopg2
from psycopg2 import Error

def connect():
    # 1. Establish the connection
    try:
        # Connect to the PostgreSQL database
        conn = psycopg2.connect(
            dbname='postgres',
            user='postgres',
            password='I4NAennDnRh1uO9z2y0k6pMfAf',
            host='localhost',
            port='5432')

        cursor = conn.cursor()
        print("Connection successful.")
        return conn,cursor
    
    except psycopg2.Error as e:
        # Handle any errors during connection or query execution
        print(f"\nAn error occurred: {e}")

def execute_query(cursor, conn, sql_query, fetch = False):
    cursor.execute(sql_query)
    if fetch == True:
        results = cursor.fetchall()
        return results
    conn.commit()


In [49]:
sql_query_string = """
CREATE TABLE test.sales (
    transaction_id INTEGER,
    product_id INTEGER,
    sale_date DATE,
    quantity INTEGER,
    price NUMERIC(10, 2) -- Use NUMERIC or DECIMAL for precise currency values
)
"""

conn_cursor = connect()

Connection successful.


In [ ]:
execute_query(cursor = conn_cursor[1], conn = conn_cursor[0], sql_query = sql_query_string)

In [51]:
conn_cursor[0].close
conn_cursor[1].close

<function cursor.close()>

In [52]:
conn_cursor = connect()

Connection successful.


In [53]:
sql_query_string = """
INSERT INTO test.sales (transaction_id, product_id, sale_date, quantity, price) VALUES
(1, 101, '2025-10-14', 5, 19.99),
(2, 102, '2025-10-14', 1, 250.00),
(3, 101, '2026-01-30', 3, 19.99),
(4, 103, '2026-03-15', 10, 5.50),
(5, 102, '2026-10-16', 2, 245.00);
"""

In [54]:
%%timeit
execute_query(cursor = conn_cursor[1], conn = conn_cursor[0], sql_query = sql_query_string)

295 μs ± 7.59 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [55]:
conn_cursor[0].close
conn_cursor[1].close

<function cursor.close()>

In [56]:
conn_cursor = connect()

Connection successful.


So now, we've inserted our data and effectively stored it in the database. DuckDB stores this data temporarily on volatile memory, unless we specify that it should save such data to disk. PostgreSQL works to store data on disk right away. This distinct difference occurs because DuckDB is an embedded database, designed to work within other processes, whereas PostgreSQL is designed to work as a database server software all on its own (which other technologies can then connect to). 

We can however, specify that DuckDB also write to disk using a "persistent" setting. Regardless, we can see that Postgres is slightly faster for loading the data: we will come back to this later, and explore this concept in the future.

Let's query the data just to see what's there:

In [ ]:
execute_query(cursor = conn_cursor[1], conn = conn_cursor[0], sql_query = "SELECT * FROM test.sales;", fetch = True)

In [60]:
conn_cursor[0].close
conn_cursor[1].close

[(1, 101, datetime.date(2025, 10, 14), 5, Decimal('19.99')),
 (2, 102, datetime.date(2025, 10, 14), 1, Decimal('250.00')),
 (3, 101, datetime.date(2026, 1, 30), 3, Decimal('19.99')),
 (4, 103, datetime.date(2026, 3, 15), 10, Decimal('5.50')),
 (5, 102, datetime.date(2026, 10, 16), 2, Decimal('245.00')),
 (1, 101, datetime.date(2025, 10, 14), 5, Decimal('19.99')),
 (2, 102, datetime.date(2025, 10, 14), 1, Decimal('250.00')),
 (3, 101, datetime.date(2026, 1, 30), 3, Decimal('19.99')),
 (4, 103, datetime.date(2026, 3, 15), 10, Decimal('5.50')),
 (5, 102, datetime.date(2026, 10, 16), 2, Decimal('245.00')),
 (1, 101, datetime.date(2025, 10, 14), 5, Decimal('19.99')),
 (2, 102, datetime.date(2025, 10, 14), 1, Decimal('250.00')),
 (3, 101, datetime.date(2026, 1, 30), 3, Decimal('19.99')),
 (4, 103, datetime.date(2026, 3, 15), 10, Decimal('5.50')),
 (5, 102, datetime.date(2026, 10, 16), 2, Decimal('245.00')),
 (1, 101, datetime.date(2025, 10, 14), 5, Decimal('19.99')),
 (2, 102, datetime.date(

## Part 2: Exploring Internal Metadata (The "Data about Data")
What about the metadata layer? Databases don't just store your data; they also store information about your data. This metadata is crucial for the query engine to work correctly. We're going to explore this later in the assignment, but let's begin with just an overview about what DuckDB and Postgres know by querying their internal "data catalogs" about our sales table.

### DuckDB Metadata Query:

In [75]:
%%sql
CALL pragma_table_info('sales');



,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,INTEGER,False,None,False
1,1,product_id,INTEGER,False,None,False
2,2,sale_date,DATE,False,None,False
3,3,quantity,INTEGER,False,None,False
4,4,price,DOUBLE,False,None,False


### PostgreSQL Metadata Query

In [78]:
conn_cursor = connect()

sql_query_string = """
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'sales';
"""

execute_query(cursor = conn_cursor[1], conn = conn_cursor[0], sql_query = sql_query_string, fetch = True)



Connection successful.


[('transaction_id', 'integer'),
 ('product_id', 'integer'),
 ('sale_date', 'date'),
 ('quantity', 'integer'),
 ('price', 'numeric')]

In [79]:
conn_cursor[0].close
conn_cursor[1].close

<function cursor.close()>

So, they can see details about the types of data we're storing, and what is expected of the table as a whole. This prevents errors and other issues that can occur when working with the database in practice. Let's see why in addition to storage and querying this is so important, by really digging deeper and seeing what happens when we get rid of the metadata system as a whole. 


## Part 3: Externalizing Metadata with Parquet and Data Catalogs

Now we can really understand why metadata shines - we can actually remove it from the equation in this step. What if we separate the metadata from the data and the query engine? This is the core idea behind modern data lakehouse architecture. Here, data is stored in an open format like Parquet or Avro, and metadata is stored in a catalog like Apache Iceberg. These are open source methods and standards for storing data, as well as managing it's metadata. Parquet storage was originally based off of a paper on Dremel (the system behind Goolge BigQuery), and Avro was designed within the Apache Hadoop project. You can read more about them below. 

Parquet: https://parquet.apache.org/

Avro: https://avro.apache.org/

Iceberg: https://iceberg.apache.org/

All of these open source standards/projects are currently managed by the Apache Foundation.

Let's try using DuckDB's feature which allows us to read and write to files directly, bypassing the standard database storage that DuckDB comes built with.

In [82]:
%%sql
COPY sales TO 'sales_data.parquet' (FORMAT parquet);

,Count
0,4055


Now, instead of a database managing everything, we have separate components:

First, we'll work with the storage layer: The sales_data.parquet file sitting in a folder or drive.

Let's query the Parquet files:

In [83]:
%%sql
SELECT * FROM 'sales_data.parquet';

,transaction_id,product_id,sale_date,quantity,price
0,1,101,2025-10-14,5,19.99
1,2,102,2025-10-14,1,250.00
2,3,101,2026-01-30,3,19.99
3,4,103,2026-03-15,10,5.50
4,5,102,2026-10-16,2,245.00
...,...,...,...,...,...
4050,1,101,2025-10-14,5,19.99
4051,2,102,2025-10-14,1,250.00
4052,3,101,2026-01-30,3,19.99
4053,4,103,2026-03-15,10,5.50


Note that this reads the file directly, and at first glance, everything appears as it was before. There is one catch though: suppose for example, we want to add to this file. We can do one of two things:

- Add a new file
- Re-write the existing file

Both are quite troublesome and involve some work. With the first option, this might be faster, but it misses out on benefits provided by compression across these two files. Some repeated values in the second file will be stored twice (in the first and the latter). 

What if we want to change the data type in the file? We would have to do the same thing again. Moreover, what if we wanted to insert a row, or update an existing row? we run into the same issues once again. Not only do these problems arise, but we also run into a bigger challenge: ACID gaurantees. We will come back to this later, but for now it is sufficient to understand that: if we want to read all the parquet or avro files in one folder, there is no real system that gaurantees across all the writers to this folder, that the data types, columns, etc., have to be atomic, consistent, isolated, or durable. Without ACID gaurantees, certain statements, such as INSERT, or UPDATE, don't really work. How can you insert data into a column, if you have no idea what type it is, for example? You would need some type of consistency gaurantee. 

For example, files could be randomly deleted, in between the parquet or avro files we could have cat videos uploaded to that drive, or we could have random code scripts executing on the same file, making conflicting changes to it. This would be challenging to any software system we want to build, that should be robust (if you want to learn more about these issues, we highly recommend reading Designing Data Intensive Applications, a survey book on this topic).

In order to manage all of this, we need a system to store data about data - what we call metadata. Now that we're writing and readying directly to files instead of DuckDB's own internal storage system, we're going to miss all of this, and we'll need to substitute something for it.

In addition to DuckDB's own system, another option that is particularly popular, is Apache Iceberg.

Apache Iceberg is an open table format designed for large analytic datasets in data lakes. It's not a storage system or a query engine; instead, it's a specification that defines how to manage the files that make up a large table. Think of it as a sophisticated index or "table of contents" for your data files (like Parquet or ORC) sitting in cloud storage like Amazon S3. Iceberg allows different query engines like Spark, Trino, and Flink to work with the same data reliably and efficiently. It also enables them to use certain statements that require ACID gaurantees such as INSERT, or UPDATE. 

Now, we'll setup Apache Iceberg in the next steps:

In [ ]:
import os
import duckdb
from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, IntegerType, StringType, FloatType
import pandas as pd
import pyarrow as pa

# --- Configuration ---
WAREHOUSE_DIR = "iceberg_catalog_duckdb"
CATALOG_DB_PATH = f"{WAREHOUSE_DIR}/catalog.db"
NAMESPACE = "dev_sales"
TABLE_IDENTIFIER = f"{NAMESPACE}.daily_summary"

# Create the directory if it doesn't exist
os.makedirs(WAREHOUSE_DIR, exist_ok=True)

# 1. Load the PyIceberg Catalog (using SQLite for metadata)
catalog = load_catalog(
    "local_duckdb",
    type="sql",
    uri=f"sqlite:///{CATALOG_DB_PATH}",
    warehouse=f"file://{WAREHOUSE_DIR}",
)
print(f"PyIceberg Catalog loaded: {catalog.name}")

# 2. Initialize DuckDB Connection
# Create an in-memory DuckDB connection
con = duckdb.connect()

# 3. Register the PyIceberg Catalog with DuckDB
# The 'pyiceberg' extension and the 'INSTALL/LOAD' commands are required
con.sql("INSTALL iceberg;")
con.sql("LOAD iceberg;")
con.sql(f"""
    CREATE OR REPLACE SECRET iceberg_key (
        TYPE DUCKDB_INTERNAL,
        PROVIDER DUCKDB_ICEBERG,
        CATALOG 'local_duckdb'
    );
""")
# Use the PyIceberg catalog object to register the catalog in DuckDB
con.sql(f"""
    CREATE OR REPLACE TABLE catalog.iceberg_catalog (
        TYPE ICEBERG,
        CATALOG_TYPE PYICEBERG,
        CATALOG_HOLDER pyiceberg,
        CATALOG_NAME '{catalog.name}'
    );
""")
print("\nDuckDB registered with PyIceberg catalog 'local_duckdb'.")

In [88]:
# 1. Create a Namespace (database) using the Python API
catalog.create_namespace_if_not_exists(NAMESPACE)

# 2. Create the Iceberg Table
iceberg_schema = Schema(
    NestedField(field_id=1, name="transaction_id", field_type=IntegerType(), required=True),
    NestedField(field_id=2, name="product_id", field_type=IntegerType(), required=True),
    NestedField(field_id=3, name="quantity", field_type=IntegerType(), required=True),
    NestedField(field_id=4, name="price", field_type=FloatType(), required=True),
)

table = catalog.create_table_if_not_exists(
    identifier=TABLE_IDENTIFIER,
    schema=iceberg_schema,
)
print(f"\nIceberg Table created: {TABLE_IDENTIFIER}")


Iceberg Table created: dev_sales.daily_summary


In [ ]:
# 3. Prepare Sample Data (from your original request)
data = pd.DataFrame({
    'transaction_id': [1, 2, 3, 4, 5],
    'product_id': [101, 102, 101, 103, 102],
    'quantity': [5, 1, 3, 10, 2],
    'price': [19.99, 250.00, 19.99, 5.50, 245.00]
})

data = data.astype({
    'transaction_id': 'int32',
    'product_id': 'int32',
    'quantity': 'int32',
    'price': 'float32'
})

arrow_schema = iceberg_schema.as_arrow()

arrow_table = pa.Table.from_pandas(data, schema=arrow_schema, preserve_index=False)

# 4. Append Data using PyIceberg API
table.append(arrow_table)
print("Data successfully appended to the Iceberg table via PyIceberg.")

Next, let's query the table - this time, we won't even use DuckDB, we'll just use PyIceberg to do a straight read right into Pandas, a totally different Python library for querying and working with data.  

In [93]:
# Assuming the 'catalog' and 'table' objects are loaded from previous steps.
# table = catalog.load_table(TABLE_IDENTIFIER)

from pyiceberg.expressions import GreaterThan, EqualTo

# --- A. Simple Full Scan ---
print("\n--- PyIceberg API Full Scan Results (as Pandas) ---")
df_full_scan = table.scan().to_pandas()
print(df_full_scan)

# --- B. Scan with Filter (Predicate Pushdown) ---
# Find transactions where quantity is greater than 2
print("\n--- PyIceberg API Filtered Scan Results ---")
filtered_scan = table.scan(
    row_filter=GreaterThan("quantity", 2),
    selected_fields=["transaction_id", "product_id", "quantity"] # Only fetch these columns
)

df_filtered = filtered_scan.to_pandas()
print(df_filtered)


--- PyIceberg API Full Scan Results (as Pandas) ---
   transaction_id  product_id  quantity   price
0               1         101         5   19.99
1               2         102         1  250.00
2               3         101         3   19.99
3               4         103        10    5.50
4               5         102         2  245.00

--- PyIceberg API Filtered Scan Results ---
   transaction_id  product_id  quantity
0               1         101         5
1               3         101         3
2               4         103        10


C:\Users\ktoto\AppData\Local\Programs\Python\Python313\Lib\site-packages\pyiceberg\avro\decoder.py:185: UserWarning: Falling back to pure Python Avro decoder, missing Cython implementation
  warnings.warn("Falling back to pure Python Avro decoder, missing Cython implementation")


See how this allows us to use ACID statements, and allows different query engines to access the same table. Pretty neat right?

See how this also guards us from making type errors, prevents uploading of random files into the db folder, and so on. There are other benefits as well, which we will review next class as we learn more about ACID. For example, using a catalog for some query engines enables speedups: to find the right data, query engines must perform slow and expensive "list directory" operations on thousands or millions of files and folders especially in cloud storage to see where each piece of data lives. But Iceberg stores all that metadata for the query engine, so it's not necessary if you're using a metadata system.



    

### Part 4: The Bring Your Own "Universe" 

So now, we have a "Bring your own Query Engine" sort of setup. We also have a "Bring your own Data Storage Format" setup. 

Overlal, we've used:

- Metadata Layer (Catalog): An Apache Iceberg or DuckLake catalog that stores metadata: the file path, table schema (column names/types), and statistics (e.g., min/max values in a column).
- Query Engine: A tool like DuckDB, Spark, or Trino that reads the metadata from the catalog to find the right data files and then executes the query.
- Storage Format: In our case, column store Parquet format, so that we can read quickly and aggregate at speed. 

What's left? What if there were other metadata storage formats?! And in fact, DuckDB comes with two - it's own internal system, which you can choose to use and completely ignore metadata management - or DuckLake!

#### DuckLake
This system is sort of like Iceberg. It manages metadata for you. However, unlike Iceberg, which stores metadata in specific files (and it does this the same way everytime), DuckLake allows you to store metadata in *many* different systems. That includes, regular files, or SQLite files, or PostgreSQL - where metadata is stored as records in a metadata table itself! Even MySQL is an option. Essentially, you can think of Ducklake as DuckDB - but instead of storing data in the typical DuckDB database format, it allows you to store data in files such as Parquet, and metadata in many other systems! 

Read more about DuckLake here: https://ducklake.select/

%%sql
INSTALL ducklake;

#### DuckLake with File Metadata:

In [ ]:
%%sql
ATTACH 'ducklake:metadata.ducklake' AS my_ducklake;
USE my_ducklake;

#### DuckLake with PostgreSQL Metadata:

In [ ]:
%%sql
INSTALL postgres;
ATTACH 'ducklake:postgres:dbname=ducklake_catalog user=postgres password=[PASSWORD] host=localhost' AS my_ducklake_postgres
     (DATA_PATH 'data_files/');
USE my_ducklake_postgres;

Immediately after attaching the database, you can create tables! Unlike the first time we created tables with DuckDB's internal system, this time the data is stored as files, but metadata is stored in Postgres. 

In [ ]:
%%sql
CREATE TABLE TESTING_DUCKLAKE (
    ID INTEGER,
    TESTING VARCHAR,
    TESTING_LAUNCH_DATE DATE
);

At this point, you can practically bring your own universe when it comes to data tools! Whatever query engine you need for your use case, whatever data storage format, and so on. The flexibility with the modern "Lake House" structure (often what this is referred to in practice) is unbeatable.

You may want to use this sort of setup, if you're more interested in integrating different tools and sources, instead of going with a one-size-fits-all approach (i.e. a single database) to your data infrastructure. 


| type | DuckLake | Iceberg |
| - | - | - |
data options | Avro, Parquet, etc. | Avro, Parquet, etc. |
metadata options |Files, PostgreSQL, MySQL | Files |


### Stand Out!
Task (Thought Exercise):

- What is the main advantage of decoupling storage, metadata, and query engine? (Hint: Think about flexibility).
- If you wanted to update the data type of the price column from FLOAT to DECIMAL(10, 2), which component(s) would you need to interact with in this new, decoupled system?


The most interesting part of this is that you can choose your query engine and your data storage to fit your needs. This means, you can use Spark, or Polars, or DuckDB - the latter allows you to run DuckLake, which instead of running DuckDB as a database, allows you to choose your storage format etc. etc. avoid lock in plug and play, customization, etc.